# POCUS-AI: Introduction to Ultrasound Image Analysis

This notebook demonstrates basic usage of the POCUS-AI package for analyzing ultrasound images using radiomics and self-supervised learning approaches.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import POCUS-AI modules
import sys
sys.path.append('..')  # Add parent directory to path
from src.pocus_ai.radiomics import extract_features
from src.pocus_ai.ssl import ContrastiveLearning

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Preprocess Sample Data

For this demonstration, we'll create a synthetic ultrasound image. In practice, you would load real DICOM or other medical image formats.

In [ ]:
# Create a synthetic ultrasound image
def create_synthetic_ultrasound(size=(256, 256)):
    # Create base image with gaussian noise
    image = np.random.normal(0, 1, size)
    
    # Add some circular structures to simulate tissue
    x, y = np.ogrid[:size[0], :size[1]]
    center = (size[0]//2, size[1]//2)
    mask = (x - center[0])**2 + (y - center[1])**2 <= (size[0]//4)**2
    image[mask] += 2
    
    # Normalize to [0, 1]
    image = (image - image.min()) / (image.max() - image.min())
    
    return image

# Generate sample image
sample_image = create_synthetic_ultrasound()

# Display the image
plt.figure(figsize=(8, 8))
plt.imshow(sample_image, cmap='gray')
plt.title('Synthetic Ultrasound Image')
plt.axis('off')
plt.show()

## 2. Radiomics Feature Extraction

Next, we'll demonstrate how to extract radiomics features from the image using POCUS-AI's radiomics module.

In [ ]:
# Extract radiomics features
try:
    features = extract_features(sample_image)
    print("Extracted features:", features)
except NotImplementedError:
    print("Note: Feature extraction not yet implemented in the package.")
    print("When implemented, it will extract features such as:")
    print("- First-order statistics (mean, variance, skewness, etc.)")
    print("- Shape-based features")
    print("- Texture features (GLCM, GLRLM, etc.)")

## 3. Self-Supervised Learning

Finally, let's demonstrate how to use the self-supervised learning module for contrastive learning on ultrasound images.

In [ ]:
# Create a simple encoder (for demonstration)
import torch.nn as nn

class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.output_dim = 32 * 64 * 64  # Depends on input size
        
    def forward(self, x):
        return self.encoder(x)

# Initialize the contrastive learning model
encoder = SimpleEncoder()
ssl_model = ContrastiveLearning(encoder)

# Create a batch of images (normally would load from dataset)
batch_size = 4
images = torch.tensor(np.stack([create_synthetic_ultrasound() for _ in range(batch_size)]))
images = images.float().unsqueeze(1)  # Add channel dimension

# Get embeddings
with torch.no_grad():
    embeddings = ssl_model(images)

print(f"Generated embeddings shape: {embeddings.shape}")
print("These embeddings can be used for downstream tasks like clustering or classification.")